# Section 1 – Core Classes
### 1. Import + Graph + Ambulance classes

In [11]:
import heapq
from collections import defaultdict
import random

class CityGraph:
    def __init__(self):
        self.adj = defaultdict(list)  # node -> list of (neighbor, weight)

    def add_edge(self, u, v, w):
        self.adj[u].append((v, w))
        self.adj[v].append((u, w))

In [12]:
class Ambulance:
    def __init__(self, aid, location):
        self.aid = aid
        self.location = location
        self.available = True

### 2. DispatchSystem (efficient single-Dijkstra version)

In [13]:
class DispatchSystem:
    def __init__(self, graph, ambulances):
        self.graph = graph
        self.ambulances = ambulances

    def _dijkstra_all_from(self, src):
        pq = [(0, src)]
        dist = {src: 0}
        while pq:
            d, u = heapq.heappop(pq)
            if d != dist.get(u, float('inf')):
                continue
            for v, w in self.graph.adj.get(u, ()):
                nd = d + w
                if nd < dist.get(v, float('inf')):
                    dist[v] = nd
                    heapq.heappush(pq, (nd, v))
        return dist

    def assign_ambulance(self, incident_loc, urgency=1):
        dist_from_incident = self._dijkstra_all_from(incident_loc)
        best = None
        for amb in self.ambulances:
            if not amb.available:
                continue
            d = dist_from_incident.get(amb.location, float('inf'))
            if d == float('inf'):
                continue
            pr = (-urgency, d, amb.aid)
            if best is None or pr < best[0]:
                best = (pr, amb, d)
        if best is None:
            return None
        _, amb, d = best
        amb.available = False
        return amb, d

### 3. Random graph generator + setup

In [14]:
def make_random_graph(n_nodes=20, edge_prob=0.2, max_w=10):
    G = CityGraph()
    for i in range(n_nodes):
        for j in range(i + 1, n_nodes):
            if random.random() < edge_prob:
                w = random.randint(1, max_w)
                G.add_edge(i, j, w)
    return G

random.seed(0)
G = make_random_graph(30, edge_prob=0.12)
ambs = [Ambulance(i, random.randrange(0, 30)) for i in range(5)]
ds = DispatchSystem(G, ambs)

### 4. Simulation (optional)

In [15]:
def simulate(dispatch_system, n_incidents=50, node_range=None, seed=0):
    random.seed(seed)
    node_range = node_range or list(dispatch_system.graph.adj.keys())
    stats = {'assigned': 0, 'unreachable': 0, 'total_distance': 0.0}
    for amb in dispatch_system.ambulances:
        amb.available = True
    for _ in range(n_incidents):
        loc = random.choice(node_range)
        out = dispatch_system.assign_ambulance(loc, urgency=1)
        if out is None:
            stats['unreachable'] += 1
        else:
            amb, d = out
            stats['assigned'] += 1
            stats['total_distance'] += d
            amb.available = True
    stats['avg_distance'] = (
        stats['total_distance'] / stats['assigned'] if stats['assigned'] else float('inf')
    )
    return stats

stats = simulate(ds, n_incidents=100)
print(stats)


{'assigned': 100, 'unreachable': 0, 'total_distance': 418.0, 'avg_distance': 4.18}


### 5. Unit tests

In [16]:
# Test 1
G = CityGraph()
G.add_edge(0, 1, 5)
G.add_edge(1, 2, 3)
ambs = [Ambulance(1, 0), Ambulance(2, 2)]
ds = DispatchSystem(G, ambs)
assert ds.assign_ambulance(1) is not None

# Test 2
G2 = CityGraph()
G2.add_edge(0, 1, 2)
ambs2 = [Ambulance(1, 2)]
ds2 = DispatchSystem(G2, ambs2)
assert ds2.assign_ambulance(0) is None

# Test 3
G3 = CityGraph()
G3.add_edge(0, 1, 1)
ambs3 = [Ambulance(1, 0), Ambulance(2, 0)]
ds3 = DispatchSystem(G3, ambs3)
assert ds3.assign_ambulance(1)[0].aid == 1

print("unit tests passed")

unit tests passed


# Section 2 – Using Real CSV Data
### 1. Load data and build the graph

In [17]:
import pandas as pd

# Load CSVs
nodes_df = pd.read_csv("nodes.csv")
edges_df = pd.read_csv("edges.csv")
ambulances_df = pd.read_csv("ambulances.csv")
traffic_df = pd.read_csv("traffic.csv")
emergencies_df = pd.read_csv("emergencies.csv")

# Create the graph
G = CityGraph()
for _, row in edges_df.iterrows():
    u, v, w = int(row["source"]), int(row["target"]), float(row["weight"])
    # adjust by current traffic if traffic.csv has extra delay or multiplier
    traffic_factor = 1.0
    if "traffic_level" in traffic_df.columns:
        traffic_factor = 1 + traffic_df.loc[
            (traffic_df["edge_id"] == row.name), "traffic_level"
        ].fillna(0).values[0]
    G.add_edge(u, v, w * traffic_factor)


ModuleNotFoundError: No module named 'pandas'

### 2. Create ambulances

In [ ]:
ambs = [
    Ambulance(int(row["ambulance_id"]), int(row["node_id"]))
    for _, row in ambulances_df.iterrows()
]

### 3. Create the DispatchSystem

In [ ]:
ds = DispatchSystem(G, ambs)

### 4. Assign ambulances to real emergencies

In [ ]:
results = []
for _, row in emergencies_df.iterrows():
    incident_loc = int(row["node_id"])
    urgency = int(row["urgency"]) if "urgency" in emergencies_df.columns else 1
    assigned = ds.assign_ambulance(incident_loc, urgency)
    if assigned:
        amb, dist = assigned
        results.append((row["emergency_id"], amb.aid, dist))
    else:
        results.append((row["emergency_id"], None, None))

results_df = pd.DataFrame(results, columns=["emergency_id", "ambulance_id", "distance"])
print(results_df.head())

### 5. Save or analyze results

In [ ]:
results_df.to_csv("assignments_output.csv", index=False)
print("Saved assignments_output.csv")